# Phase 3 Failures — Run Record

This notebook runs the Phase 3 fault injection sweep and displays results.
Prefer running via CLI (`scripts/run_phase3.py`) for full sweeps — this notebook
is for inspection and documentation.

In [1]:
import yaml
import json
import sys
from pathlib import Path
import nest_asyncio
nest_asyncio.apply()

from dotenv import load_dotenv
load_dotenv(Path("../../.env"))

sys.path.insert(0, str(Path("../../src")))
sys.path.insert(0, str(Path("../../")))

config_path = Path("../../config/phase3.yaml")
raw = yaml.safe_load(config_path.read_text())
print("Config:")
print(yaml.dump(raw, default_flow_style=False))

Config:
artifacts:
  output_dir: artifacts/
fault_types:
- after_seconds: 3.0
  name: timeout
- fail_on_call: 1
  name: malformed_json
- name: checkpoint_loss
- name: context_overflow
  token_limit: 200
llm:
  api_key: ${OPENAI_API_KEY}
  base_url: ${OPENAI_BASE_URL}
  max_tokens: 1024
  model: openai/openai/gpt-4o-mini
  request_timeout: 120
  temperature: 0.0
mcp:
  workspace_root: workspace/
run:
  inter_run_delay_s: 1
  n_runs: 5
  run_id_prefix: failures



In [2]:
# Load existing Phase 3 artifacts (run via CLI, not re-run here)
from analysis.aggregate import load_runs
import pandas as pd

df = load_runs("../../artifacts/", prefix="failures")
print(f"Loaded {len(df)} Phase 3 runs")
display(df[["run_id", "failure_type", "recovery_mode", "total_latency_s", "output_quality"]].head(30))

Loaded 20 Phase 3 runs


,run_id,failure_type,recovery_mode,total_latency_s,output_quality
0,failures_checkpointloss_2ae5be2f,checkpoint_loss,manual,36.386569,1.0
1,failures_checkpointloss_43440c34,checkpoint_loss,manual,37.763846,1.0
2,failures_checkpointloss_4ef123b4,checkpoint_loss,manual,40.264925,1.0
3,failures_checkpointloss_92e048ba,checkpoint_loss,manual,39.770985,1.0
4,failures_checkpointloss_c32e8615,checkpoint_loss,manual,34.892739,1.0
5,failures_contextoverflow_60169c83,context_overflow,manual,0.005259,0.0
6,failures_contextoverflow_d6d6f547,context_overflow,manual,0.003351,0.0
7,failures_contextoverflow_e6f8b77f,context_overflow,manual,0.002260,0.0
8,failures_contextoverflow_ec738d88,context_overflow,manual,0.003097,0.0
9,failures_contextoverflow_f92a1873,context_overflow,manual,0.003251,0.0


In [3]:
# Recovery summary table
if not df.empty:
    summary = df.groupby("failure_type").agg(
        runs=("run_id", "count"),
        automatic=("recovery_mode", lambda x: (x == "automatic").sum()),
        manual=("recovery_mode", lambda x: (x == "manual").sum()),
        avg_recovery_time_s=("recovery_time_s", "mean"),
        avg_output_quality=("output_quality", "mean"),
        avg_latency_s=("total_latency_s", "mean"),
    ).round(3)
    display(summary)
    print(f"\nTotal: {len(df)} runs across {df['failure_type'].nunique()} fault types")

,runs,automatic,manual,avg_recovery_time_s,avg_output_quality,avg_latency_s
failure_type,,,,,,
checkpoint_loss,5,0,5,0.001,1.0,37.816
context_overflow,5,0,5,0.000,0.0,0.003
malformed_json,5,5,0,0.001,0.0,0.002
timeout,5,0,5,0.001,0.0,3.004



Total: 20 runs across 4 fault types


In [4]:
# Sample metrics from one run per fault type
if not df.empty:
    for ft in df["failure_type"].unique():
        sample = df[df["failure_type"] == ft].iloc[0]
        run_dir = Path(f"../../artifacts/{sample['run_id']}")
        metrics_file = run_dir / "metrics.json"
        if metrics_file.exists():
            print(f"\n=== {ft} ===")
            print(json.dumps(json.loads(metrics_file.read_text()), indent=2))


=== checkpoint_loss ===
{
  "run_id": "failures_checkpointloss_2ae5be2f",
  "nodes": [],
  "total_latency_s": 36.38656945800176,
  "total_prompt_tokens": 0,
  "total_completion_tokens": 0,
  "step_trace": [],
  "error": "State checkpoint written but lost on read-back (injected fault). Output files may exist on disk but LangGraph state has no record of them.",
  "message_count": null,
  "state_size_bytes": null,
  "failure_type": "checkpoint_loss",
  "recovery_time_s": 0.0008380413055419922,
  "output_quality": 1.0,
  "recovery_mode": "manual",
  "notes": "Output files exist on disk but LangGraph state has no record \u2014 demonstrates checkpoint asymmetry: files written before state update lost.",
  "gcc_sim": false
}

=== context_overflow ===
{
  "run_id": "failures_contextoverflow_60169c83",
  "nodes": [],
  "total_latency_s": 0.005259290977846831,
  "total_prompt_tokens": 0,
  "total_completion_tokens": 0,
  "step_trace": [],
  "error": "Estimated context 410 tokens exceeds limit 2